In [1]:
import os
from pathlib import Path

os.getcwd()
mb_dir = Path(os.getcwd()).parent.parent.parent
os.chdir(mb_dir)
data_dir = str(mb_dir.parent / "data")
root_dir = str(mb_dir.parent)

In [2]:
import xarray as xr 
import numpy as np

from examples.paper_figures.data_utils import (
    get_model_dfs, get_plot_metrics, save_data,
    load_wyi, get_climatological_dfs, load_wyi,
    YEAR_RANGES, EXTENDED_YEARS, YEAR_RANGES_COM
)

from monsoonbench.metrics import (
    ClimatologyOnsetMetrics
)

from monsoonbench.visualization import create_model_comparison_table

c_metrics = ClimatologyOnsetMetrics()

# Figure 4 year ranges
YEAR_RANGES_COM = {
    "AIFS": np.arange(2004, 2022),
    "IFS": np.arange(2004, 2022),
    "FuXi": np.arange(2004, 2022),
    "Graphcast": np.arange(2004, 2022),
    "GenCast": np.arange(2019, 2022),
    "FuXi-S2S": np.arange(2004, 2022),
    "NGCM": np.arange(2004, 2022),
}

config = {
    "years": np.arange(2019, 2025),
    "extended_years": np.concatenate((np.arange(1965, 1979), np.arange(2019, 2025))),  # Extended period for analysis
    "common_years": np.arange(2004, 2022),
    "imd_folder": f"{data_dir}/imd_rainfall_data/4p0",  # Ground truth rainfall data (4x4 degrees)
    "thres_file": f"{data_dir}/imd_onset_threshold/mwset4x4.nc4",  # Threshold for the onset of the monsoon (4x4 degrees)
    "shpfile_path": f"{data_dir}/ind_map_shpfile/india_shapefile.shp",  # Shapefile of India
    "output_dir": f"{root_dir}/output",  # Directory to save data files
}

config2 = {
    "imd_folder": f"{data_dir}/imd_rainfall_data/4p0",  # Ground truth rainfall data (4x4 degrees)
    "thres_file": f"{data_dir}/imd_onset_threshold/mwset4x4.nc4",  # Threshold for the onset of the monsoon (4x4 degrees)
    "shpfile_path": f"{data_dir}/ind_map_shpfile/india_shapefile.shp",  # Shapefile of India
    "output_dir": f"{root_dir}/output",  # Directory to save data files
    "years": np.concatenate((np.arange(1965, 1979), np.arange(2019, 2025)))  # Extended period for analysis
}


config3 = {
    "imd_folder": f"{data_dir}/imd_rainfall_data/4p0",  # Ground truth rainfall data (4x4 degrees)
    "thres_file": f"{data_dir}/imd_onset_threshold/mwset4x4.nc4",  # Threshold for the onset of the monsoon (4x4 degrees)
    "shpfile_path": f"{data_dir}/ind_map_shpfile/india_shapefile.shp",  # Shapefile of India
    "output_dir": f"{root_dir}/output",  # Directory to save data files
    "years": np.arange(2004, 2022)  # Extended period for analysis
}

model_paths = {
    "IFS": f"{data_dir}/rainfall_4p0/IFS_S2S",
    "AIFS":  f"{data_dir}/rainfall_4p0/AIFS",
    "FuXi": f"{data_dir}/rainfall_4p0/FuXi",
    "Graphcast": f"{data_dir}/rainfall_4p0/GraphCast",
    "GenCast": f"{data_dir}/rainfall_4p0/GenCast",
    "FuXi-S2S": f"{data_dir}/rainfall_4p0/FuXi_S2S",
    "NGCM": f"{data_dir}/rainfall_4p0/NeuralGCM"
}

### Loading fig 3 ground truth data

In [3]:
import scipy.io as sio
weekly_file = f"{root_dir}/fig_data/5day_forecastwindow_cmz_2019_2024.mat"
data = sio.loadmat(weekly_file)
mae_cmz = data['mae_cmz']  # Shape should be (6, 8) for 6 time periods, 8 models
far_cmz = data['far_cmz']  # Shape should be (6, 8)
mr_cmz = data['mr_cmz']    # Shape should be (4, 8) for 4 weeks
std_er = data['std_er']    # Standard errors for MAE

print(data["model_str"])
mae_cmz
gt_mae = []
for r in mae_cmz:
    gt_mae.append(r[0])

[[array(['clim'], dtype='<U4')]
 [array(['ifs'], dtype='<U3')]
 [array(['aifs'], dtype='<U4')]
 [array(['fuxi'], dtype='<U4')]
 [array(['graphcast'], dtype='<U9')]
 [array(['gencast'], dtype='<U7')]
 [array(['fuxis2s'], dtype='<U7')]
 [array(['ngcm51'], dtype='<U6')]]


Generating climatoligical window data

In [ ]:
DEFAULT_WINDOW_BINS: list[tuple[int, int]] = [
    (1, 6),
    (6, 11),
    (11, 16),
    (16, 21),
    (21, 26),
    (26, 31),
]

def get_clim_window_data(
    config: dict[str, str],
):
    clim_data = []

    for (lower, upper) in DEFAULT_WINDOW_BINS:
        if lower < 11:
            tol_days = 2
        elif lower < 21:
            tol_days = 3
        else:
            tol_days = 5
        
        multi_yr_metrics, multi_onset_dy = c_metrics.compute_climatology_baseline_multiple_years(        
            years=config["years"],
            imd_folder=config["imd_folder"],
            thres_file=config["thres_file"],
            tolerance_days=tol_days, #Tolerance window
            verification_window=lower, 
            forecast_days=upper,
            max_forecast_day=upper,
            mok=True,
            onset_window=5,
            mok_month=6,
            mok_day=2,
        )
        clim_plot_data = c_metrics.create_spatial_far_mr_mae(
            multi_yr_metrics, dict.fromkeys(config["years"], multi_onset_dy)
        )
        
        clim_data.append(clim_plot_data)
        
    return clim_data

Computing climatological onset reference...
Computing climatological onset from 124 years: 1901-2024
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1901.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1901-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1902.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1902-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1903.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1903-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1904.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1904-06-02) as start date for onset detection
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3

In [ ]:
import pandas as pd
clim_window_data = {}
clim_data = get_clim_window_data(config)

for i, (lower, upper) in enumerate(DEFAULT_WINDOW_BINS):
    clim_window_data[f"{lower}-{upper}"] = clim_data[i]


rep_df = create_model_comparison_table(clim_window_data)
pd.DataFrame({
    "Correct MAE (clim)": gt_mae,
    "Reproduced MAE (clim)": rep_df["cmz_mae_mean_days"].values
})

,Correct MAE (clim),Reproduced MAE (clim)
0,4.523810,4.523810
1,4.938757,4.938757
2,5.878704,5.878704
3,5.878704,5.878704
4,6.188889,6.188889
5,6.864815,6.864815


In [ ]:
from monsoonbench.metrics import (
    ClimatologyOnsetMetrics,
    DeterministicOnsetMetrics,
    ProbabilisticOnsetMetrics,
)

import pandas as pd
windows = [1,6,11,16,21,26]

def get_window_data(
    model_paths: dict[str, str],
    year_ranges: dict[str, list[int]],
    config: dict[str, str],
    lower_window: int = 1,
) -> tuple[dict[str, pd.DataFrame], dict[str, xr.DataArray]]:
    """Get model dataframes and onset data arrays for a given set of model paths, year ranges, and forecast period.

    Args:
        model_paths: Dictionary of model names and their file paths.
        year_ranges: Dictionary of model names and their year ranges.
        config: Dictionary of configuration parameters.
        lower_window: Lower bound of the verification window.
    """

    upper_window = lower_window + 5
    metrics = ProbabilisticOnsetMetrics()
    d_metrics = DeterministicOnsetMetrics()

    model_dfs = {}
    model_onsets = {}

    # Validate verification window and tolerance days
    if lower_window < 11:
        tol_days = 2
    elif lower_window < 21:
        tol_days = 3
    else:
        tol_days = 5

    # Compute metrics for each model
    for model_name, model_fp in model_paths.items():
        try:
            probabilistic_df, onset_da_dict = metrics.compute_metrics_multiple_years(
                years=year_ranges[model_name],
                imd_folder=config["imd_folder"],
                thres_file=config["thres_file"],
                model_forecast_dir=model_fp,
                tolerance_days=tol_days, #tolerance for metric calculations
                verification_window=lower_window, #start of window
                forecast_days=upper_window, #lower_window + 5
                max_forecast_day=upper_window, #lower_window + 5
                mok=True,
                onset_window=5,
                mok_month=6,
                mok_day=2,
            )

        except Exception:
            probabilistic_df, onset_da_dict = d_metrics.compute_metrics_multiple_years(
                years=year_ranges[model_name],
                imd_folder=config["imd_folder"],
                thres_file=config["thres_file"],
                model_forecast_dir=model_fp,
                tolerance_days=tol_days,
                verification_window=lower_window,
                forecast_days=upper_window,
                max_forecast_day=upper_window,
                mok=True,
                onset_window=5,
                mok_month=6,
                mok_day=2,
            )

        model_dfs[model_name] = probabilistic_df
        model_onsets[model_name] = onset_da_dict

    return model_dfs, model_onsets


model_paths = {
    "IFS": f"{data_dir}/rainfall_4p0/IFS_S2S",
    "AIFS":  f"{data_dir}/rainfall_4p0/AIFS",
    "FuXi": f"{data_dir}/rainfall_4p0/FuXi",
    "Graphcast": f"{data_dir}/rainfall_4p0/GraphCast",
    "GenCast": f"{data_dir}/rainfall_4p0/GenCast",
    "FuXi-S2S": f"{data_dir}/rainfall_4p0/FuXi_S2S",
    "NGCM": f"{data_dir}/rainfall_4p0/NeuralGCM"
}

mp = {
    "FuXi": f"{data_dir}/rainfall_4p0/FuXi",
}

window_dfs = []

for start_date in windows:
    f3_df, f3_onsets = get_window_data(model_paths, YEAR_RANGES, config, lower_window=start_date)
    window_data = {}
    for model_name in model_paths.keys():
        plot_probabilistic_metrics = c_metrics.create_spatial_far_mr_mae(
            f3_df[model_name], f3_onsets[model_name]
        )
        window_data[model_name] = plot_probabilistic_metrics

    window_df = create_model_comparison_table(window_data)
    window_dfs.append(window_df)



Processing year 2019
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\2019.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (2019-06-02) as start date for onset detection
Processing 26 init times x 8 lats x 9 lons...
Using MOK (6/2 filter) for onset detection
Only processing forecasts initialized before observed onset dates
Requiring ≥50% of 11 members to have onset for ensemble onset
Processing init time 1/26: 2019-05-02
Processing init time 6/26: 2019-05-20
Processing init time 11/26: 2019-06-06
Processing init time 16/26: 2019-06-24
Processing init time 21/26: 2019-07-11
Processing init time 26/26: 2019-07-29

Processing Summary:
Total potential initializations: 1872
Skipped (no observed onset): 962
Skipped (initialized after observed onset): 394
Valid initializations processed: 516
Ensemble onsets found (≥50% members): 68
Ensemble onset rate: 0.132
Note: Only onsets on or after 6/2 were counted due to MOK flag
Computing on

In [ ]:
far = []
mae = []
std_er = []
mr = []
for i in range(len(window_dfs)):
    far_win = [rep_df.iloc[i]["cmz_far_pct"]]
    mae_win = [rep_df.iloc[i]["cmz_mae_mean_days"]]
    std_er_win = [rep_df.iloc[i]["cmz_mae_se_days"]]
    mr_win = [rep_df.iloc[i]["cmz_mr_pct"]]

    far_win += window_dfs[i]["cmz_far_pct"].values.tolist()
    mae_win += window_dfs[i]["cmz_mae_mean_days"].values.tolist()
    std_er_win += window_dfs[i]["cmz_mae_se_days"].values.tolist()
    mr_win += window_dfs[i]["cmz_mr_pct"].values.tolist()

    far.append(np.array(far_win))
    mae.append(np.array(mae_win))
    std_er.append(np.array(std_er_win))
    mr_.append(np.array(mr_win))

out_dict = {
    "far_cmz": np.array(far),
    "mae_cmz": np.array(mae),
    "model_str" = np.array(["clim", "ifs", "aifs", "fuxi", "graphcast",
                "gencast", "fuxis2s", "ngcm51"]),
    "std_er": np.array(std_er),
    "mr_cmz": np.array(mr)
}

,cmz_mae_mean_days,cmz_mae_se_days,cmz_far_pct,cmz_mr_pct,overall_mae_mean_days,overall_far_pct,overall_mr_pct
model,,,,,,,
FuXi,1.563161,0.263142,1.764323,43.111111,4.708323,3.446468,54.661204


In [ ]:
def save_data(mat_dict: dict[str, np.ndarray], output_dir: str, save_path: str) -> None:
    """Save data to a .mat file.

    Args:
        mat_dict: Dictionary of data to save.
        output_dir: Directory to save the data.
        save_path: Path to save the data.
    """
    out_path = f"{output_dir}/{save_path}.mat"
    savemat(out_path, mat_dict)
    print("Saved to:", out_path)
    return

save_data(out_dict, f"{root_dir}/output", "5day_forecastwindow_cmz_2019_2024.mat")

In [6]:
import monsoonbench.spatial.regions as r 
cmz_lon, cmz_lat = r.get_cmz_polygon_coords(4)
grid_lats = np.unique(test_clim_forecast[2019]["lat"])
grid_lons = np.unique(test_clim_forecast[2019]["lon"])
r.points_inside_polygon(cmz_lon, cmz_lat, grid_lons, grid_lats)


(array([[False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False],
        [False, False,  True,  True,  True, False, False, False],
        [False,  True,  True,  True,  True, False, False, False],
        [False,  True,  True,  True, False, False, False, False],
        [False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False]]),
 array([76., 80., 84., 72., 76., 80., 84., 72., 76., 80.]),
 array([20., 20., 20., 24., 24., 24., 24., 28., 28., 28.]))